# E32 --- o alerta que todo mundo usa, com o par declarado

A prática vigia a mudança que ainda vem com uma receita que tem nome e literatura: **a variância e a
autocorrelação da janela sobem antes da transição chegar** --- aqui com a assimetria junto, que é a
terceira perna da mesma família e o objeto que o vigia da direção já mede em blocos. A receita nunca
declarou o par que o capítulo do orçamento cobra de todo alarme: **com que frequência ela soa quando
nada muda, e quanto demora quando algo muda**.

Esta medição submete a receita à régua do livro, e o controle é o **posto do corte** --- o alarme mais
fundo que a janela de 252 dias permite (o pior dia do último ano), medido nos mesmos mundos:

1. **calibração**: os limiares da receita saem de mundos que nunca mudam, no mesmo gasto *medido* do
   posto, em **episódios por ano** (a janela rolante arrasta o mesmo dia; dias seguidos acima são um
   alarme só);
2. **latência**: forma por forma (degrau, rampa, deriva), com o chão do mundo parado ao lado;
3. **o olho**: a calibração refeita como a prática faz --- o limiar escolhido num mundo que muda,
   sem orçamento --- e o gasto dele medido nos mundos parados;
4. **as séries reais**: o que a receita declara e o que ela gasta em três mercados, e o aviso que deu
   nos dois tombos do S&P 500.

**Simulação não vira resultado sobre o mundo** (AGENTS.md §8.5): os mundos sintéticos medem
propriedades dos instrumentos; o painel real é leitura de mercado, sem data de mudança conhecida.


In [1]:
# <- brinque com: JANELA, MUNDOS_PARADO, CASOS, HORIZONTE, MUNDO_DIAS, MUDANCA_EM, SEMENTE, SERIES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import alerta, dados, graficos, mudanca, promessa, vigia, volatilidade

RAIZ = Path.cwd()
JANELA = 252             # a janela do corte: a memória que a receita herda do livro
DIAS_UTEIS = 252
ORCAMENTO_POSTO = vigia.orcamento_minimo(JANELA)   # 1/(n+1) por dia: o mais fundo que a janela dá
MUNDOS_PARADO = 200     # mundos que nunca mudam, para calibrar (e outros tantos para o chão)
MUNDO_DIAS = 3000       # o mundo sintético, do tamanho do E02
MUDANCA_EM = 1500       # o dia em que a mudança entra
CASOS = 200             # mundos por forma, para a latência
HORIZONTE = 500         # dias depois da mudança em que se procura o alarme
OLHO_RAMPA_DIAS = 250   # a rampa que dobra ao longo de 250 dias (o padrão da casa)
SEMENTE = 61
SERIES = ("sp500.csv", "ibov.csv", "btc.csv")

SORTEIO = np.random.default_rng(SEMENTE)
SORTEIO_OLHO = np.random.default_rng(SEMENTE + 1000)
INDICADORES = {"variancia": alerta.variancia, "autocorrelacao": alerta.autocorrelacao,
               "assimetria": alerta.assimetria}
FORMAS = {"degrau": mudanca.degrau, "rampa": mudanca.rampa, "deriva": mudanca.deriva}
print("frevolab %s | janela %d | posto no orcamento %.6f/dia | %d mundos parados | %d casos por forma"
      % (frevolab.VERSAO, JANELA, ORCAMENTO_POSTO, MUNDOS_PARADO, CASOS))

frevolab 0.1.0 | janela 252 | posto no orcamento 0.003953/dia | 200 mundos parados | 200 casos por forma


## Painel 1 --- a calibração: o gasto do posto é a régua

Cada estatística rolante vira alarme por um limiar, e o limiar é bisseccionado nos mundos parados até
gastar **o mesmo que o posto gasta, medido** --- em episódios por ano, não em dias: o posto também
agrupa dias, e é o episódio a unidade de quem ouve o alarme.

In [2]:
series_paradas = [pd.Series(mudanca.estavel(MUNDO_DIAS, SORTEIO)) for _ in range(MUNDOS_PARADO)]
posto_nulos = np.stack([vigia.dispara(s, JANELA, ORCAMENTO_POSTO).to_numpy() for s in series_paradas])
gasto_posto = alerta.orcamento(posto_nulos.astype(float), 0.5)["episodios_por_ano"]

nulos = {nome: np.stack([fun(s, JANELA).loc[JANELA:].to_numpy() for s in series_paradas])
         for nome, fun in INDICADORES.items()}
limiares, gastos = {}, {"posto": gasto_posto}
for nome, matriz in nulos.items():
    limiares[nome] = alerta.limiar_do_orcamento(matriz, gasto_posto)
    gastos[nome] = alerta.orcamento(matriz, limiares[nome])["episodios_por_ano"]

print("%16s %14s %18s" % ("instrumento", "episodios/ano", "limiar"))
print("%16s %14.3f %18s" % ("posto", gasto_posto,
                            "posto %d" % vigia.posto_do_orcamento(JANELA, ORCAMENTO_POSTO)))
for nome in INDICADORES:
    print("%16s %14.3f %18.6g" % (nome, gastos[nome], limiares[nome]))

     instrumento  episodios/ano             limiar
           posto          0.995            posto 1
       variancia          0.995        0.000112933
  autocorrelacao          0.995          0.0914647
      assimetria          0.995            0.21569


## Painel 2 --- a latência, forma por forma, com o chão do mundo parado

O mesmo orçamento, as três formas da casa, e --- ao lado de cada instrumento --- o **chão**: o dia
mediano do primeiro falso alarme em mundos que nunca mudam, contado do primeiro dia em que o alarme
existe. Latência menor que o chão do próprio instrumento é o orçamento sendo gasto na frente da
mudança, não visão dela.

In [3]:
LATENCIAS, NADAS = {}, {}
for forma, geradora in FORMAS.items():
    mundos = [pd.Series(geradora(MUNDO_DIAS, SORTEIO)) for _ in range(CASOS)]
    alarmes_da_forma = {"posto": [vigia.dispara(s, JANELA, ORCAMENTO_POSTO) for s in mundos]}
    for nome, fun in INDICADORES.items():
        alarmes_da_forma[nome] = [alerta.dispara(fun(s, JANELA), limiares[nome]) for s in mundos]
    for nome, lista in alarmes_da_forma.items():
        lat = np.array([vigia.latencia(a, MUDANCA_EM) for a in lista])
        LATENCIAS[(nome, forma)] = float(np.nanmedian(lat)) if np.isfinite(lat).any() else float("nan")
        NADAS[(nome, forma)] = 100.0 * float(np.mean(~np.isfinite(lat)))

# O chão: mundos parados novos, nunca vistos pela calibração.
series_chao = [pd.Series(mudanca.estavel(MUNDO_DIAS, SORTEIO)) for _ in range(MUNDOS_PARADO)]
alarmes_chao = {"posto": [vigia.dispara(s, JANELA, ORCAMENTO_POSTO) for s in series_chao]}
for nome, fun in INDICADORES.items():
    alarmes_chao[nome] = [alerta.dispara(fun(s, JANELA), limiares[nome]) for s in series_chao]
chao = {}
for nome, lista in alarmes_chao.items():
    primeiros = [int(a.index[np.flatnonzero(a.to_numpy())][0]) - JANELA for a in lista
                 if a.to_numpy().any()]
    chao[nome] = float(np.median(primeiros)) if primeiros else float("nan")

print("%16s %16s %16s %16s %10s" % ("instrumento", "degrau", "rampa", "deriva", "chão"))
for nome in ["posto"] + list(INDICADORES):
    print("%16s %16s %16s %16s %10.0f" % (
        nome,
        "%.0f (%.0f%%)" % (LATENCIAS[(nome, "degrau")], NADAS[(nome, "degrau")]) if np.isfinite(LATENCIAS[(nome, "degrau")]) else "não vê",
        "%.0f (%.0f%%)" % (LATENCIAS[(nome, "rampa")], NADAS[(nome, "rampa")]) if np.isfinite(LATENCIAS[(nome, "rampa")]) else "não vê",
        "%.0f (%.0f%%)" % (LATENCIAS[(nome, "deriva")], NADAS[(nome, "deriva")]) if np.isfinite(LATENCIAS[(nome, "deriva")]) else "não vê",
        chao[nome]))

     instrumento           degrau            rampa           deriva       chão
           posto           8 (0%)          80 (0%)         166 (0%)        158
       variancia          12 (0%)          84 (0%)        452 (14%)        475
  autocorrelacao        116 (14%)        271 (12%)        395 (24%)        486
      assimetria          32 (8%)        171 (12%)        400 (22%)        514


## Painel 3 --- a calibração no olho, como a prática faz

Ninguém bissecciona limiar em duzentos mundos parados. O que se faz é olhar **um** mundo que muda,
ver o indicador subir, e pôr a linha onde o desenho parece alarmante. Aqui a linha é posta no meio da
rampa de um único mundo sorteado --- e mede-se o que ela gasta nos duzentos mundos parados, e se ela
ainda vê a rampa em mundos que não foram olhados.

In [4]:
mundo_olho = pd.Series(mudanca.rampa(MUNDO_DIAS, SORTEIO_OLHO))
DIA_OLHO = MUDANCA_EM + OLHO_RAMPA_DIAS // 2
LIMIAR_OLHO = float(alerta.variancia(mundo_olho, JANELA).loc[DIA_OLHO])

orcamento_olho = alerta.orcamento(nulos["variancia"], LIMIAR_OLHO)
mundos_rampa_novos = [pd.Series(mudanca.rampa(MUNDO_DIAS, SORTEIO)) for _ in range(CASOS)]
alarmes_olho = [alerta.dispara(alerta.variancia(s, JANELA), LIMIAR_OLHO) for s in mundos_rampa_novos]
lat_olho = np.array([vigia.latencia(a, MUDANCA_EM) for a in alarmes_olho])
print("limiar do olho: %.6g (o da calibração era %.6g)" % (LIMIAR_OLHO, limiares["variancia"]))
print("gasto em mundo parado: %.2f anos por alarme (%.3f episódios/ano)"
      % (orcamento_olho["anos_por_alarme"], orcamento_olho["episodios_por_ano"]))
print("rampa em mundos novos: mediana %.0f dias | %.1f%% não viram"
      % (np.nanmedian(lat_olho), 100.0 * float(np.mean(~np.isfinite(lat_olho)))))

limiar do olho: 0.000132244 (o da calibração era 0.000112933)
gasto em mundo parado: 90.87 anos por alarme (0.011 episódios/ano)
rampa em mundos novos: mediana 132 dias | 0.0% não viram


## Painel 4 --- as séries reais: o declarado contra o gasto, e os dois tombos

A calibração dos mundos parados é gaussiana na escala de cada série. O mercado tem agrupamento de
oscilação --- e é lição do vigia da direção que **o orçamento se lê da medição, e não da álgebra**:
aqui a diferença entre o gasto declarado e o gasto medido é parte do resultado. Nos dois tombos do
S&P 500, o primeiro alarme dentro do tombo: quantos dias depois do topo, e quanto da queda já estava
pago.

In [5]:
GASTOS_REAIS = {}
for arquivo in SERIES:
    preco_r = dados.carregar_serie(arquivo)
    retornos_r = volatilidade.retornos_log(preco_r).dropna()
    sigma_r = float(retornos_r.std())
    paradas_r = [pd.Series(SORTEIO.normal(0.0, sigma_r, retornos_r.size)) for _ in range(MUNDOS_PARADO)]
    matriz_posto_r = np.stack([vigia.dispara(s, JANELA, ORCAMENTO_POSTO).to_numpy() for s in paradas_r])
    gasto_alvo = alerta.orcamento(matriz_posto_r.astype(float), 0.5)["episodios_por_ano"]
    medidos = {}
    alarme_posto_r = vigia.dispara(retornos_r, JANELA, ORCAMENTO_POSTO)
    medidos["posto"] = (int((alarme_posto_r & ~alarme_posto_r.shift(1, fill_value=False)).sum())
                        / (alarme_posto_r.size / DIAS_UTEIS))
    for nome, fun in INDICADORES.items():
        matriz_r = np.stack([fun(s, JANELA).loc[JANELA:].to_numpy() for s in paradas_r])
        limiar_r = alerta.limiar_do_orcamento(matriz_r, gasto_alvo)
        alarme_r = alerta.dispara(fun(retornos_r, JANELA), limiar_r)
        medidos[nome] = (int((alarme_r & ~alarme_r.shift(1, fill_value=False)).sum())
                         / (alarme_r.size / DIAS_UTEIS))
    GASTOS_REAIS[arquivo] = {"alvo": gasto_alvo, "medidos": medidos, "sigma": sigma_r}
    print("%-11s sigma %.4f | alvo %.2f/ano | " % (arquivo, sigma_r, gasto_alvo)
          + " | ".join("%s %.2f" % (n, v) for n, v in medidos.items()))

sp500.csv   sigma 0.0121 | alvo 0.98/ano | posto 1.13 | variancia 0.31 | autocorrelacao 0.35 | assimetria 0.62


ibov.csv    sigma 0.0169 | alvo 1.00/ano | posto 0.91 | variancia 0.32 | autocorrelacao 0.63 | assimetria 1.27


btc.csv     sigma 0.0349 | alvo 0.99/ano | posto 0.91 | variancia 0.43 | autocorrelacao 0.06 | assimetria 0.79


In [6]:
preco_sp = dados.carregar_serie("sp500.csv")
retornos_sp = volatilidade.retornos_log(preco_sp).dropna()
topo_antigo = preco_sp.loc["2007-01-01":"2008-06-30"].idxmax()
fundo_antigo = preco_sp.loc["2008-01-01":"2009-06-30"].idxmin()
topo_recente = preco_sp.loc["2019-01-01":"2020-03-31"].idxmax()
fundo_recente = preco_sp.loc["2020-01-01":"2020-06-30"].idxmin()

paradas_sp = [pd.Series(SORTEIO.normal(0.0, float(retornos_sp.std()), retornos_sp.size))
              for _ in range(MUNDOS_PARADO)]
matriz_posto_sp = np.stack([vigia.dispara(s, JANELA, ORCAMENTO_POSTO).to_numpy() for s in paradas_sp])
gasto_sp = alerta.orcamento(matriz_posto_sp.astype(float), 0.5)["episodios_por_ano"]
matriz_var_sp = np.stack([alerta.variancia(s, JANELA).loc[JANELA:].to_numpy() for s in paradas_sp])
LIMIAR_VAR_SP = alerta.limiar_do_orcamento(matriz_var_sp, gasto_sp)

TOMBOS = {}
for nome in ("posto", "variancia"):
    alarme_t = (vigia.dispara(retornos_sp, JANELA, ORCAMENTO_POSTO) if nome == "posto"
                else alerta.dispara(alerta.variancia(retornos_sp, JANELA), LIMIAR_VAR_SP))
    comecos_t = alarme_t.index[alarme_t.to_numpy() & ~alarme_t.shift(1, fill_value=False).to_numpy()]
    TOMBOS[nome] = {}
    for rotulo, topo, fundo in (("antigo", topo_antigo, fundo_antigo),
                                ("recente", topo_recente, fundo_recente)):
        dentro = comecos_t[(comecos_t >= topo) & (comecos_t <= fundo)]
        if len(dentro):
            dia = dentro[0]
            pago_t = vigia.prejuizo_pago(preco_sp, dia)
            TOMBOS[nome][rotulo] = {"data": str(dia.date()), "atraso": (dia - topo).days,
                                    "pago_pct": 100.0 * pago_t["fracao_paga"]}
        else:
            TOMBOS[nome][rotulo] = None
        print("%s | tombo %s: topo %s fundo %s -> %s" % (
            nome, rotulo, topo.date(), fundo.date(), TOMBOS[nome][rotulo]))

posto | tombo antigo: topo 2007-10-09 fundo 2009-03-09 -> {'data': '2008-09-09', 'atraso': 336, 'pago_pct': 38.33359777184794}
posto | tombo recente: topo 2020-02-19 fundo 2020-03-23 -> {'data': '2020-02-24', 'atraso': 5, 'pago_pct': 13.950816954570183}
variancia | tombo antigo: topo 2007-10-09 fundo 2009-03-09 -> {'data': '2008-07-08', 'atraso': 273, 'pago_pct': 32.79805483149772}
variancia | tombo recente: topo 2020-02-19 fundo 2020-03-23 -> {'data': '2020-03-12', 'atraso': 22, 'pago_pct': 78.82568093715997}


## As figuras

In [7]:
# Figura 1: a latência de cada instrumento nas três formas, no mesmo orçamento.
FORMAS_EIXO = ["degrau", "rampa", "deriva"]
NOMES_FIG = {"posto": "posto", "variancia": "variância",
             "autocorrelacao": "autocorrelação", "assimetria": "assimetria"}
largura = 0.2
fig, ax = plt.subplots(figsize=(9.4, 5.2))
for k, nome in enumerate(NOMES_FIG):
    valores = [LATENCIAS[(nome, f)] for f in FORMAS_EIXO]
    y_pos = [i - 1.5 * largura + k * largura for i in range(len(FORMAS_EIXO))]
    barras = [min(v, HORIZONTE) if np.isfinite(v) else HORIZONTE for v in valores]
    ax.barh(y_pos, barras, height=largura, label=NOMES_FIG[nome],
            alpha=0.45 if nome != "posto" else 1.0)
    for y, v, f in zip(y_pos, valores, FORMAS_EIXO):
        rotulo = "%.0f" % v if np.isfinite(v) else "não vê"
        if not np.isfinite(v) or NADAS[(nome, f)] > 0:
            rotulo += " (%.0f%% não vê)" % NADAS[(nome, f)]
        ax.text((min(v, HORIZONTE) if np.isfinite(v) else HORIZONTE) + 6, y, rotulo,
                va="center", fontsize=7.5)
ax.set_yticks(range(len(FORMAS_EIXO)))
ax.set_yticklabels(["degrau (dobra)", "rampa (dobra em 250 dias)", "deriva (média desliza)"])
ax.invert_yaxis()
ax.set_xlim(0, HORIZONTE + 260)
ax.set_xlabel("dias entre a mudança e o alarme (a linha pontilhada é o horizonte de %d dias)" % HORIZONTE)
ax.axvline(HORIZONTE, color="0.4", lw=0.8, ls=":")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
graficos.salvar(fig, "E32_alerta", 1)
plt.close(fig)

In [8]:
# Figura 2: o S&P 500 inteiro, os dois tombos, e os alarmes dos dois instrumentos.
episodios_posto = vigia.dispara(retornos_sp, JANELA, ORCAMENTO_POSTO)
comecos_posto = episodios_posto.index[episodios_posto.to_numpy()
                                      & ~episodios_posto.shift(1, fill_value=False).to_numpy()]
estatistica_sp = alerta.variancia(retornos_sp, JANELA)
episodios_var = alerta.dispara(estatistica_sp, LIMIAR_VAR_SP)
comecos_var = episodios_var.index[episodios_var.to_numpy()
                                  & ~episodios_var.shift(1, fill_value=False).to_numpy()]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9.4, 6.4), sharex=True,
                               gridspec_kw={"height_ratios": [2.0, 1.2]})
ax1.plot(preco_sp.index, preco_sp.to_numpy(), lw=0.7, color="0.15")
ax1.set_yscale("log")
for topo, fundo in ((topo_antigo, fundo_antigo), (topo_recente, fundo_recente)):
    ax1.axvspan(topo, fundo, color="0.75", alpha=0.45, zorder=0)
ax1.plot(comecos_posto, [preco_sp.min() * 0.75] * len(comecos_posto), "|", color="tab:blue",
         ms=9, label="posto")
ax1.plot(comecos_var, [preco_sp.min() * 0.75] * len(comecos_var), "|", color="tab:orange",
         ms=9, label="variância")
ax1.set_ylabel("preço (escala log)")
ax1.legend(loc="upper left", fontsize=9)
ax2.plot(estatistica_sp.index, estatistica_sp.to_numpy() * 1e4, lw=0.6, color="0.3")
ax2.axhline(LIMIAR_VAR_SP * 1e4, color="tab:red", lw=1.0, ls="--")
ax2.plot(comecos_var, [LIMIAR_VAR_SP * 1e4] * len(comecos_var), "|", color="tab:red", ms=8)
ax2.set_ylabel("variância da janela (em 10^-4)")
ax2.set_xlabel("a faixa cinza é o tombo; o traço vermelho é o limiar calibrado")
fig.tight_layout()
graficos.salvar(fig, "E32_alerta", 2)
plt.close(fig)

## Leitura visual das figuras

Feita contra o PNG de cada figura, com o código ao lado --- a lição da auditoria de figuras: o código
não decide posição relativa, o olho sim.

**Figura 1** --- três grupos de quatro barras (degrau, rampa, deriva), o posto em azul cheio e as
três pernas da receita em meio-tom; o número em cada barra é a mediana, e onde houve mundos que não
alarmaram o percentual vem escrito ao lado; a linha pontilhada é o horizonte de 500 dias. O que o
olho vê antes da conta: no degrau a barra do posto é a menor de todas e as da autocorrelação e da
assimetria são as únicas que atravessam a marca dos 100 dias; na rampa as barras do posto e da
variância são quase a mesma barra (80 contra 84); na deriva nenhuma barra começa antes de 160 dias e
as três da receita passam de 390 --- mais perto do horizonte do que do começo.

**Figura 2** --- o preço em escala log em cima, com as duas faixas cinzas dos tombos e as marcas dos
dois instrumentos rente à base; a variância da janela embaixo, com o limiar calibrado em traço
vermelho. O que o olho confirma: a curva de baixo cruza o limiar em picos curtos dentro das faixas
cinzas --- e também no começo dos anos 2000, que não está em faixa nenhuma, porque os dois tombos
escolhidos não cobrem aquele. As marcas dos dois instrumentos se acumulam nas mesmas regiões de
turbulência; dentro da faixa de 2020 a marca do posto cai nos primeiros dias e a da variância vem
depois, já perto do fim da faixa.

In [9]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "alerta_janela": JANELA,
    "alerta_mundos_parado": MUNDOS_PARADO,
    "alerta_casos": CASOS,
    "alerta_horizonte_dias": HORIZONTE,
    "alerta_mundo_dias": MUNDO_DIAS,
    "alerta_mudanca_dia": MUDANCA_EM,
    "alerta_posto_episodios_ano": round(float(gasto_posto), 3),
    "alerta_olho_dia": DIA_OLHO,
}
for nome in INDICADORES:
    resultado["alerta_%s_episodios_ano" % nome] = round(float(gastos[nome]), 3)
    resultado["alerta_%s_limiar" % nome] = float(limiares[nome])
    resultado["alerta_%s_chao_dias" % nome] = round(float(chao[nome]), 1)
resultado["alerta_posto_chao_dias"] = round(float(chao["posto"]), 1)
# O dia é a mediana dos mundos que viram; quando nenhum viu, entra o horizonte --- censura
# declarada, e o livro lê o número sempre com o nada_pct ao lado.
for (nome, forma), v in LATENCIAS.items():
    resultado["alerta_%s_%s_dias" % (nome, forma)] = round(float(v), 1) if np.isfinite(v) else int(HORIZONTE)
    resultado["alerta_%s_%s_nada_pct" % (nome, forma)] = round(float(NADAS[(nome, forma)]), 2)
resultado["alerta_olho_limiar"] = float(LIMIAR_OLHO)
resultado["alerta_olho_anos_por_alarme"] = round(float(orcamento_olho["anos_por_alarme"]), 2)
resultado["alerta_olho_episodios_ano"] = round(float(orcamento_olho["episodios_por_ano"]), 3)
mediana_olho = float(np.nanmedian(lat_olho))
resultado["alerta_olho_rampa_dias"] = round(mediana_olho, 1) if np.isfinite(mediana_olho) else int(HORIZONTE)
resultado["alerta_olho_rampa_nada_pct"] = round(100.0 * float(np.mean(~np.isfinite(lat_olho))), 2)
ROTULOS_SERIES = {"sp500.csv": "sp", "ibov.csv": "ibov", "btc.csv": "btc"}
for arquivo, dados_r in GASTOS_REAIS.items():
    rotulo = ROTULOS_SERIES[arquivo]
    for nome, v in dados_r["medidos"].items():
        resultado["alerta_real_%s_%s_episodios_ano" % (rotulo, nome)] = round(float(v), 3)
for nome, tombos_d in TOMBOS.items():
    for rotulo, dado in tombos_d.items():
        if dado:
            resultado["alerta_tombo_%s_%s_atraso_dias" % (rotulo, nome)] = int(dado["atraso"])
            resultado["alerta_tombo_%s_%s_pago_pct" % (rotulo, nome)] = round(dado["pago_pct"], 2)

# O que sai do laboratório e o que o livro cita: medida que o livro não usa é medida morta.
CITADAS_NO_LIVRO = (
    "alerta_assimetria_chao_dias", "alerta_assimetria_degrau_dias", "alerta_assimetria_deriva_dias",
    "alerta_assimetria_deriva_nada_pct", "alerta_assimetria_rampa_dias",
    "alerta_autocorrelacao_chao_dias", "alerta_autocorrelacao_degrau_dias",
    "alerta_autocorrelacao_deriva_dias", "alerta_autocorrelacao_deriva_nada_pct",
    "alerta_autocorrelacao_rampa_dias", "alerta_casos", "alerta_horizonte_dias", "alerta_janela",
    "alerta_mundos_parado", "alerta_olho_anos_por_alarme", "alerta_olho_episodios_ano",
    "alerta_olho_rampa_dias", "alerta_posto_chao_dias", "alerta_posto_degrau_dias",
    "alerta_posto_deriva_dias", "alerta_posto_episodios_ano", "alerta_posto_rampa_dias",
    "alerta_real_btc_assimetria_episodios_ano", "alerta_real_btc_autocorrelacao_episodios_ano",
    "alerta_real_btc_posto_episodios_ano", "alerta_real_btc_variancia_episodios_ano",
    "alerta_real_ibov_assimetria_episodios_ano", "alerta_real_ibov_autocorrelacao_episodios_ano",
    "alerta_real_ibov_posto_episodios_ano", "alerta_real_ibov_variancia_episodios_ano",
    "alerta_real_sp_assimetria_episodios_ano", "alerta_real_sp_autocorrelacao_episodios_ano",
    "alerta_real_sp_posto_episodios_ano", "alerta_real_sp_variancia_episodios_ano",
    "alerta_tombo_antigo_posto_atraso_dias", "alerta_tombo_antigo_posto_pago_pct",
    "alerta_tombo_antigo_variancia_atraso_dias", "alerta_tombo_antigo_variancia_pago_pct",
    "alerta_tombo_recente_posto_atraso_dias", "alerta_tombo_recente_posto_pago_pct",
    "alerta_tombo_recente_variancia_atraso_dias", "alerta_tombo_recente_variancia_pago_pct",
    "alerta_variancia_chao_dias", "alerta_variancia_degrau_dias", "alerta_variancia_deriva_dias",
    "alerta_variancia_deriva_nada_pct", "alerta_variancia_rampa_dias",
)
resultado = {chave: valor for chave, valor in resultado.items() if chave in CITADAS_NO_LIVRO}

caminho = Path("lab/resultados/E32_alerta.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E32_alerta.json gravado | 47 grandezas
